In [1]:
import sys; sys.path.append("../src")
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.ensemble import IsolationForest
from data import load, TIME, AMOUNT, SPLIT
from metrics import evaluate

tx, feats = load()
train_mask    = tx[TIME] <= SPLIT
test_lab_mask = (tx[TIME] > SPLIT) & tx["y"].notna()

X = tx[feats].fillna(tx.loc[train_mask, feats].median())   # fill gaps using train only
y = tx.loc[test_lab_mask, "y"].to_numpy()
print(len(feats), "features | train rows:", train_mask.sum())

182 features | train rows: 136265


In [2]:
iso = IsolationForest(n_estimators=200, random_state=42, n_jobs=-1)
iso.fit(X[train_mask])                      # all train rows, labels never used

tx["iso_score"] = -iso.score_samples(X)     # flip sign: higher = more anomalous
thr = tx.loc[train_mask, "iso_score"].quantile(0.99)

amt = tx[AMOUNT].fillna(0)
rng = np.random.default_rng(42)
results = [
    evaluate("Random (reference)", y, rng.random(len(y)), 0.99),
    evaluate("Amount baseline", y, amt[test_lab_mask].to_numpy(),
             amt[train_mask].quantile(0.99)),
    evaluate("Isolation Forest (all features)", y,
             tx.loc[test_lab_mask, "iso_score"].to_numpy(), thr),
]
pd.DataFrame(results)

,model,P@50,P@100,P@500,PR-AUC,flag_rate_%
0,Random (reference),0.02,0.05,0.07,0.066,1.14
1,Amount baseline,0.00,0.00,0.00,0.044,1.19
2,Isolation Forest (all features),0.00,0.00,0.00,0.037,6.10


In [3]:
local = [c for c in feats if c.lower().startswith("local")]
print(len(local), "local features")

iso_l = IsolationForest(n_estimators=200, random_state=42, n_jobs=-1)
iso_l.fit(X.loc[train_mask, local])
tx["iso_local"] = -iso_l.score_samples(X[local])

results.append(evaluate("Isolation Forest (local only)", y,
               tx.loc[test_lab_mask, "iso_local"].to_numpy(),
               tx.loc[train_mask, "iso_local"].quantile(0.99)))
pd.DataFrame(results)

93 local features


,model,P@50,P@100,P@500,PR-AUC,flag_rate_%
0,Random (reference),0.02,0.05,0.07,0.066,1.14
1,Amount baseline,0.00,0.00,0.00,0.044,1.19
2,Isolation Forest (all features),0.00,0.00,0.00,0.037,6.10
3,Isolation Forest (local only),0.00,0.00,0.00,0.038,4.52


In [4]:
def add(name, col):
    results.append(evaluate(name, y, tx.loc[test_lab_mask, col].to_numpy(),
                            tx.loc[train_mask, col].quantile(0.99)))

X_log = np.sign(X) * np.log1p(np.abs(X))

iso_log = IsolationForest(n_estimators=200, random_state=42, n_jobs=-1)
iso_log.fit(X_log[train_mask])
tx["iso_log"] = -iso_log.score_samples(X_log)
add("Isolation Forest (log features)", "iso_log")
pd.DataFrame(results)

,model,P@50,P@100,P@500,PR-AUC,flag_rate_%
0,Random (reference),0.02,0.05,0.07,0.066,1.14
1,Amount baseline,0.00,0.00,0.00,0.044,1.19
2,Isolation Forest (all features),0.00,0.00,0.00,0.037,6.10
3,Isolation Forest (local only),0.00,0.00,0.00,0.038,4.52
4,Isolation Forest (log features),0.00,0.00,0.00,0.036,6.05


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

m = train_mask.to_numpy()
sc  = StandardScaler().fit(X_log[m])
pca = PCA(n_components=20, random_state=42).fit(sc.transform(X_log[m]))
Z   = pca.transform(sc.transform(X_log))

km = KMeans(n_clusters=10, n_init=10, random_state=42).fit(Z[m])
tx["km_dist"] = km.transform(Z).min(axis=1)
add("K-Means distance (log, PCA-20)", "km_dist")
pd.DataFrame(results)

,model,P@50,P@100,P@500,PR-AUC,flag_rate_%
0,Random (reference),0.02,0.05,0.070,0.066,1.14
1,Amount baseline,0.00,0.00,0.000,0.044,1.19
2,Isolation Forest (all features),0.00,0.00,0.000,0.037,6.10
3,Isolation Forest (local only),0.00,0.00,0.000,0.038,4.52
4,Isolation Forest (log features),0.00,0.00,0.000,0.036,6.05
5,"K-Means distance (log, PCA-20)",0.02,0.02,0.012,0.037,10.64


In [6]:
K, PER = 50, 4
km50 = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Z[m])
tx["cluster"] = km50.predict(Z)

lab_train = tx[train_mask & tx["y"].notna()]
verified = (lab_train.sample(frac=1, random_state=42)
            .groupby("cluster").head(PER))
print("Verified labels used:", len(verified))

prior = verified["y"].mean()
g = verified.groupby("cluster")["y"]
risk = (g.sum() + 2 * prior) / (g.count() + 2)
tx["cluster_risk"] = tx["cluster"].map(risk).fillna(prior)

add("Cluster-then-verify (K=50, 200 labels)", "cluster_risk")
pd.DataFrame(results)

Verified labels used: 200


,model,P@50,P@100,P@500,PR-AUC,flag_rate_%
0,Random (reference),0.02,0.05,0.070,0.066,1.14
1,Amount baseline,0.00,0.00,0.000,0.044,1.19
2,Isolation Forest (all features),0.00,0.00,0.000,0.037,6.10
3,Isolation Forest (local only),0.00,0.00,0.000,0.038,4.52
4,Isolation Forest (log features),0.00,0.00,0.000,0.036,6.05
5,"K-Means distance (log, PCA-20)",0.02,0.02,0.012,0.037,10.64
6,"Cluster-then-verify (K=50, 200 labels)",0.00,0.02,0.106,0.155,0.54


In [7]:
from sklearn.metrics import average_precision_score

pr = []
for s in range(5):
    v = lab_train.sample(frac=1, random_state=s).groupby("cluster").head(PER)
    p = v["y"].mean()
    gg = v.groupby("cluster")["y"]
    r = (gg.sum() + 2 * p) / (gg.count() + 2)
    s_ = tx.loc[test_lab_mask, "cluster"].map(r).fillna(p).to_numpy()
    pr.append(average_precision_score(y, s_))

print("PR-AUC per seed:", np.round(pr, 3))
print("Mean:", round(np.mean(pr), 3), "| Std:", round(np.std(pr), 3))

PR-AUC per seed: [0.122 0.172 0.156 0.168 0.158]
Mean: 0.155 | Std: 0.017


In [8]:
test_df = tx[test_lab_mask]
check = (test_df.groupby("cluster")["y"].agg(test_illicit_rate="mean", test_labeled="size")
         .join(risk.rename("estimated_risk"))
         .join(verified.groupby("cluster").size().rename("n_verified"))
         .sort_values("estimated_risk", ascending=False))
check.head(10).round(3)

,test_illicit_rate,test_labeled,estimated_risk,n_verified
cluster,,,,
4,0.000,90,0.518,4
2,0.250,2043,0.352,4
23,1.000,17,0.352,4
39,0.000,2,0.352,4
41,0.070,157,0.185,4
22,0.018,546,0.185,4
0,0.000,2,0.018,4
7,0.000,505,0.018,4
8,0.045,376,0.018,4


In [9]:
def cluster_verify(per, seed):
    v = lab_train.sample(frac=1, random_state=seed).groupby("cluster").head(per)
    p = v["y"].mean()
    g = v.groupby("cluster")["y"]
    r = (g.sum() + 2 * p) / (g.count() + 2)
    return tx["cluster"].map(r).fillna(p), v

rows = []
for per in [2, 4, 10, 20, 50]:
    prs = []
    for s in range(5):
        sc_, v = cluster_verify(per, s)
        prs.append(average_precision_score(y, sc_[test_lab_mask].to_numpy()))
    rows.append({"labels_per_cluster": per, "total_labels": len(v),
                 "pct_of_train_labels": round(len(v) / len(lab_train) * 100, 1),
                 "PR-AUC mean": round(np.mean(prs), 3), "std": round(np.std(prs), 3)})

budget = pd.DataFrame(rows)
budget.to_csv("../results/label_budget.csv", index=False)
budget

,labels_per_cluster,total_labels,pct_of_train_labels,PR-AUC mean,std
0,2,100,0.3,0.126,0.045
1,4,200,0.7,0.155,0.017
2,10,500,1.7,0.197,0.012
3,20,998,3.3,0.199,0.007
4,50,2417,8.1,0.200,0.008


In [10]:
PER = 10
tx["cluster_risk"], verified = cluster_verify(PER, 42)
add(f"Cluster-then-verify (K=50, {len(verified)} labels)", "cluster_risk")

cluster_info = (tx.groupby("cluster").size().rename("transactions").to_frame()
                .join(verified.groupby("cluster").size().rename("n_verified"))
                .fillna({"n_verified": 0}).reset_index())
cluster_info.to_csv("../results/cluster_info.csv", index=False)
pd.DataFrame(results)

,model,P@50,P@100,P@500,PR-AUC,flag_rate_%
0,Random (reference),0.02,0.05,0.070,0.066,1.14
1,Amount baseline,0.00,0.00,0.000,0.044,1.19
2,Isolation Forest (all features),0.00,0.00,0.000,0.037,6.10
3,Isolation Forest (local only),0.00,0.00,0.000,0.038,4.52
4,Isolation Forest (log features),0.00,0.00,0.000,0.036,6.05
5,"K-Means distance (log, PCA-20)",0.02,0.02,0.012,0.037,10.64
6,"Cluster-then-verify (K=50, 200 labels)",0.00,0.02,0.106,0.155,0.54
7,"Cluster-then-verify (K=50, 500 labels)",0.00,0.09,0.280,0.190,0.54


In [11]:
Path("../results").mkdir(exist_ok=True)
tx[["txId", TIME, AMOUNT, "y", "cluster", "cluster_risk", "iso_score"]] \
  .rename(columns={"cluster_risk": "score"}) \
  .to_csv("../results/scores.csv", index=False)

In [12]:
def verify_window(lo, hi, per, seed):
    pool = tx[tx[TIME].between(lo, hi) & tx["y"].notna()]
    v = pool.sample(frac=1, random_state=seed).groupby("cluster").head(per)
    p = v["y"].mean()
    g = v.groupby("cluster")["y"]
    r = (g.sum() + 2 * p) / (g.count() + 2)
    return tx["cluster"].map(r).fillna(p), v

def mean_prauc(lo, hi, eval_mask, y_eval):
    return np.mean([average_precision_score(y_eval, verify_window(lo, hi, 10, s)[0][eval_mask].to_numpy())
                    for s in range(5)])

# Validation: "future" = steps 25-34 (still training data)
val_mask = tx[TIME].between(25, 34) & tx["y"].notna()
y_val = tx.loc[val_mask, "y"].to_numpy()
print("VALIDATION  all history (1-24):", round(mean_prauc(1, 24, val_mask, y_val), 3))
print("VALIDATION  recent only (15-24):", round(mean_prauc(15, 24, val_mask, y_val), 3))

VALIDATION  all history (1-24): 0.479
VALIDATION  recent only (15-24): 0.513


In [13]:
print("TEST  all history (1-34):", round(mean_prauc(1, 34, test_lab_mask, y), 3))
print("TEST  recent only (25-34):", round(mean_prauc(25, 34, test_lab_mask, y), 3))

TEST  all history (1-34): 0.197
TEST  recent only (25-34): 0.174


In [14]:
tx["cluster_risk"], verified = verify_window(25, 34, 10, 42)
add(f"Cluster-then-verify, recent window (K=50, {len(verified)} labels)", "cluster_risk")

cluster_info = (tx.groupby("cluster").size().rename("transactions").to_frame()
                .join(verified.groupby("cluster").size().rename("n_verified"))
                .fillna({"n_verified": 0}).reset_index())
cluster_info.to_csv("../results/cluster_info.csv", index=False)
pd.DataFrame(results)

,model,P@50,P@100,P@500,PR-AUC,flag_rate_%
0,Random (reference),0.02,0.05,0.070,0.066,1.14
1,Amount baseline,0.00,0.00,0.000,0.044,1.19
2,Isolation Forest (all features),0.00,0.00,0.000,0.037,6.10
3,Isolation Forest (local only),0.00,0.00,0.000,0.038,4.52
4,Isolation Forest (log features),0.00,0.00,0.000,0.036,6.05
5,"K-Means distance (log, PCA-20)",0.02,0.02,0.012,0.037,10.64
6,"Cluster-then-verify (K=50, 200 labels)",0.00,0.02,0.106,0.155,0.54
7,"Cluster-then-verify (K=50, 500 labels)",0.00,0.09,0.280,0.190,0.54
8,"Cluster-then-verify, recent window (K=50, 421 ...",0.00,0.02,0.238,0.185,0.54


In [15]:
Path("../results").mkdir(exist_ok=True)
tx[["txId", TIME, AMOUNT, "y", "cluster", "cluster_risk", "iso_score"]] \
  .rename(columns={"cluster_risk": "score"}) \
  .to_csv("../results/scores.csv", index=False)
pd.DataFrame(results).to_csv("../results/metrics.csv", index=False)